# 🎛️ Setting the Time Circuits
## The Art of the System Prompt

**Section 4** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### ☁️ This is the **hosted** version.
> It needs an API key but **no GPU**.
> The other version is **`04_time_circuits_local.ipynb`**.

---

In Section 2 we watched a language model predict text one token at a time. It
did that with no idea who it was, who it was talking to, or what it was for.

That is the honest default. **A model wakes up with no memory every single time
you call it.** It does not remember you, your last question, or the last ten
thousand conversations it had. Every call starts from nothing.

So how does a chatbot ever seem to have a personality?

> ### 🎯 You tell it who to be. Every time. In writing.

That instruction is the **system prompt**, and it is the single highest-leverage
thing in this entire repository.

```
DESTINATION TIME    ▸  the system prompt — who the model is
PRESENT TIME        ▸  the question you are asking right now
LAST TIME DEPARTED  ▸  where you just came from
```

Punch 1885 into the DeLorean's dashboard and it behaves like the Old West.
Punch a different system prompt into a model and you get a different assistant.

---

### What you need

| | |
|---|---|
| ⏱️ | About 5 minutes |
| 🔑 | An API key in your `.env` file |
| 🎮 | No GPU required |
| 🌐 | Internet |

---

# 🚦 LIVE DEMO PATH

## Setup

One helper, `ask()`. Give it a system prompt and a question, get an answer back.

**This is the only cell that differs between the two versions of this notebook.**
Everything below it is identical, because everything below it is about the
*prompt*, not about the machinery underneath.

In [1]:
from delorean import llm, viz

client = llm.get_api_client()
MODEL = llm.api_model()


def ask(system: str, question: str, max_tokens: int = 110) -> str:
    """Send one system prompt and one question. Get the answer back.

    Note what is NOT here: any history, any memory, any state. Each call is
    complete on its own, which is exactly how the model actually works.

    The hosted API takes the system prompt as its own argument rather than as
    a message in the list. Same idea, slightly different plumbing.
    """
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": question}],
    }
    # Omit `system` entirely when there isn't one. Sending an empty string or
    # None is not the same as not sending it.
    if system.strip():
        kwargs["system"] = system

    response = client.messages.create(**kwargs)
    return response.content[0].text.strip()


print(f"Talking to {MODEL}")

Talking to claude-haiku-4-5


---

## Act 1 — 🤷 No destination set

First, no system prompt at all. Just a question.

This is the model with nothing punched into the dashboard.

In [2]:
QUESTION = "My property assessment went up 15% this year. What should I do?"

answer = ask("", QUESTION)
print(answer)

Here are some practical options to consider:

**If you think it's wrong:**
- Review the assessment details for errors (square footage, bed/bath count, condition)
- Compare to recent sales of similar nearby properties
- File an appeal (deadline varies by location—check yours soon)
- Consider hiring an assessor if values are very high

**If the assessment is accurate:**
- Understand your actual tax impact (assessment ≠ tax bill; rates matter too)
- Budget for higher property taxes going


Perfectly reasonable. Generic, cautious, a bit shapeless — the voice of
something that does not know whether it is talking to a homeowner, an
assessor, or a lawyer.

Now let's tell it.

---

## Act 2 — 🎛️ Punch in a destination

Exactly the same question. One sentence of English added in front of it.

In [3]:
APPRAISER = (
    "You are a senior property appraiser with 25 years of experience in a "
    "county assessor's office. You explain assessment matters plainly and "
    "accurately to members of the public, and you are careful to separate "
    "what you know from what you are guessing."
)

viz.show_time_circuits(
    destination="Senior appraiser, 25 years",
    present=QUESTION,
    last_departed="(no system prompt)",
)

print(ask(APPRAISER, QUESTION))

# First Steps to Take

Here's what I'd recommend:

1. **Get the details in writing**
   - Request a copy of your assessment notice and any valuation report
   - Check that property characteristics are correct (square footage, lot size, condition, features)
   - Look for obvious errors—wrong number of bedrooms, garage size, etc.

2. **Understand the reason**
   - Your county assessor's office can tell you what drove the increase
   - Sometimes it's a re


> ### 💡 Same model. Same weights. Same question. Different answer.
>
> Nothing was retrained. Nothing was fine-tuned. Nothing was looked up. The
> only thing that changed is a sentence of plain English that the model read
> before it read your question.

---

## Act 3 — ⚖️ Three destinations at once

The clearest way to see it is side by side. Same question, three very
different instructions.

In [4]:
PERSONAS = {
    "🏛️ Senior appraiser": APPRAISER,

    "😤 Angry taxpayer": (
        "You are an angry taxpayer who is convinced the county has "
        "over-assessed your home and is wasting your money."
    ),

    "🔬 Doc Brown": (
        "You are Doc Emmett Brown, the eccentric inventor from Hill Valley, "
        "1985. You speak with wild enthusiasm and constant scientific tangents."
    ),
}

for n, (name, system_prompt) in enumerate(PERSONAS.items()):
    answer = ask(system_prompt, QUESTION)
    viz.show_html(
        viz.persona_card_html(
            name, answer,
            system_prompt=system_prompt,
            accent=viz.PERSONA_COLORS[n % len(viz.PERSONA_COLORS)],
        )
    )

> ### 💡 The system prompt is not a setting. It is the job description.
>
> This is why "prompt engineering" stopped sounding silly. That sentence is
> the difference between a model that answers like a search engine and one
> that answers like a colleague who knows your office.

---

## Act 4 — 🔍 What the model actually receives

Here is the part that surprises people, and it ties directly back to Section 2.

The system prompt is not a special channel. It is not configuration. It is not
stored anywhere. It is **text, glued onto the front of your question**, and the
whole thing is turned into tokens exactly like everything else.

In [5]:
# The hosted API keeps the system prompt in its own field, but it is still
# just text and you are still billed for it on every single call.
counted = client.messages.count_tokens(
    model=MODEL,
    system=APPRAISER,
    messages=[{"role": "user", "content": QUESTION}],
)

system_only = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)

print(f"question alone            : {system_only.input_tokens:>4} tokens")
print(f"question + system prompt  : {counted.input_tokens:>4} tokens")
print()
print(f"The system prompt costs {counted.input_tokens - system_only.input_tokens} "
      f"tokens on EVERY call.")
print("It is re-sent every time, because the model remembers nothing between calls.")

question alone            :   23 tokens
question + system prompt  :   76 tokens

The system prompt costs 53 tokens on EVERY call.
It is re-sent every time, because the model remembers nothing between calls.


> ### 🧠 This is the memory point, and it matters more than it sounds.
>
> The model did not "remember" it was an appraiser. It was **told again**, from
> scratch, in that same call. A chatbot that appears to remember your
> conversation is being re-sent the entire conversation, every single time you
> press enter.
>
> There is no state. There is only text going in.

---

## 🎬 That's Section 4

Three things to carry forward:

1. 🎛️ **The system prompt is where your office goes.** Your terminology, your
   policies, your tone, what staff may and may not rely on.
2. 🔁 **It is re-sent on every call**, and you are billed for it every time.
   There is no memory. There is only the text you send.
3. 📝 **It is plain English.** No training, no code, no vendor. Anyone in your
   office can write one, and anyone can read it and tell you it's wrong.

And notice what we just did. We took a vague, hedging model and made it sound
like **a confident 25-year expert.**

> ### ⚠️ Hold onto that, because it is about to cost us.
>
> ### ⏭️ **Next: Section 5 — Running out of Plutonium.**
> We ask our confident new expert something it cannot possibly know.

---

---

# 🔬 EXPLORE ON YOUR OWN

*Off the live path — for when you take this back to the office.*

---

## 5. 🪤 "You are a helpful assistant" does almost nothing

The most common system prompt in the world is very close to a no-op, because
being a helpful assistant is already what the model was trained to be.

In [6]:
bland = ask("You are a helpful assistant.", QUESTION)
none  = ask("", QUESTION)

print("WITH 'You are a helpful assistant':")
print(bland[:300])
print()
print("WITH NO SYSTEM PROMPT AT ALL:")
print(none[:300])

WITH 'You are a helpful assistant':
Here are some practical steps to consider:

**Understand what happened**
- Review the assessment notice for details on what changed
- Check if your property was recently reassessed or if there was a general revaluation
- Verify the property description is accurate (square footage, features, etc.)

*

WITH NO SYSTEM PROMPT AT ALL:
Here are the main things to consider:

**Understand why it increased**
- Check your local assessor's office website or contact them directly
- Was it a reassessment of your home's value, or did they find an error in previous records?
- Compare to similar homes in your area

**Your options:**

1. **D


Nearly the same answer. **All of the value is in being specific** — what the
model should know, who it is talking to, what it must refuse to do, and what
it should say when it is unsure.

---

## 6. 🏛️ A system prompt you could actually deploy

This is closer to what a real office would put in front of staff. Notice how
much of it is about **limits**, not personality.

In [7]:
OFFICE_DESK = (
    "You are the front counter of the Hill Valley County Assessor's Office. "
    "You answer in at most three sentences, in plain language, for members of "
    "the public who are not appraisers. "
    "You never guess. If the answer depends on a specific parcel, on a "
    "current value, or on county policy you have not been given, you say so "
    "plainly and tell the caller exactly where to look. "
    "You never tell anyone what their taxes will be."
)

for q in [
    "Why did my assessment go up when I haven't done anything to the house?",
    "What will my tax bill be next year?",
]:
    print(f"Q: {q}")
    print(f"A: {ask(OFFICE_DESK, q)}")
    print()

Q: Why did my assessment go up when I haven't done anything to the house?


A: Assessments can go up for several reasons even without changes to your house—like changes in the local real estate market, a correction of a previous undervaluation, or updates to property records. To find out the specific reason for your assessment increase, I'd recommend calling our office directly or visiting in person so we can look at your individual parcel and explain what changed.

Q: What will my tax bill be next year?


A: I can't tell you what your tax bill will be—that depends on your specific property value, local tax rates, and various other factors that change yearly. For your actual tax bill estimate, you'll need to contact the Assessor's Office directly or check the county tax collector's office, since they handle billing.



> 💡 The second question is the interesting one. A good system prompt makes the
> model **refuse** — and refusing correctly is a feature, not a failure.

---

## 8. ✏️ Write your own

Change the system prompt below and run it. This is the whole skill.

In [8]:
MY_SYSTEM = "You are a helpful assistant."
MY_QUESTION = "Explain what a ratio study is."

print(ask(MY_SYSTEM, MY_QUESTION, max_tokens=160))

# Ratio Study

A **ratio study** is a statistical analysis tool that compares the relationship between two related quantities to understand patterns, trends, or performance. It's commonly used in various fields to evaluate efficiency, quality, and consistency.

## Key Characteristics

- **Compares two measurable values** to show how one relates to the other
- **Expresses results** as a proportion, percentage, or rate
- **Identifies patterns** that raw numbers alone might not reveal

## Common Applications

### Business & Finance
- Profit margins, debt-to-equity ratios, return on investment
- Cost comparison across departments or time periods

### Quality Control & Manufacturing
- **Defect ratios** - comparing defective items to total


---

<div align="center">

### 🕰️ *"If my calculations are correct, when this baby hits 88 miles per hour...* 🕰️
### *you're gonna see some serious speed."*

🚗💨 🎛️ 🏘️

</div>